In [ ]:
import os
import numpy as np
from scipy.optimize import minimize, differential_evolution, LinearConstraint
from topological_insulator import Problem

structure_path = "../../../../../topological_insulator/data/structures/"
structure_name = "kagome.json"

In [ ]:
class InteractionOptimizer:
    def __init__(self, Delta_SOC=-0.2, t=-1, U=-1.6):
        # Load initial wavefunction
        self.x0 = [
            U, 
            0.1, 0, 0.1, 0, 0.8, 0,
            0, 0.1, 0, 0.1, 0, 0.8,
            0.1, 0, 0.1, 0, 0.8, 0,
        ]
        self.structure_path = structure_path
        self.structure_name = structure_name
        self.t = t
        self.Delta_SOC = Delta_SOC
        self.U = U
        os.makedirs("results", exist_ok=True)

    def _fitness(self, x):
        try:
            print("")
            problem = Problem(structure_path=structure_path, structure_name=structure_name)
            self._set_eigenvalues(problem, x)
            problem.setup(
                N_r=10,
                N_k=250,
                location="bulk",
                BZ="reduced"
            )
            problem.run(H_type="reciprocal")
            invariants = problem.hamiltonian["bulk"]["topological_invariants"]
            dE = invariants.get_band_gap(22, 21, only_dE=True)
            if np.isnan(dE) or np.isinf(dE): 
                return 1e10  # Large penalty for invalid dE
            return dE
        except Exception as e:
            print(f"Error in _fitness: {e}")
            return 1e10  # Large penalty for exceptions

    def objective(self, x):
        dE = self._fitness(x)
        print(f"x = {x}")
        print(f"dE = {dE}")
        np.savetxt(f"results/interacting_parameters_{dE:.4f}.txt", x)
        return -dE

    def get_bounds(self):
        n = len(self.x0)
        U_bounds = [(-1.6, 0)]
        occupation_bounds = [(0, 1) for _ in range(n-1)]
        return U_bounds + occupation_bounds
    
    def get_constraints(self):
        # Constraints for occupation sums, ignoring the first element (U)
        return [
            {'type': 'eq', 'fun': lambda x: np.sum(x[1:7]) - 1.0},
            {'type': 'eq', 'fun': lambda x: np.sum(x[7:13]) - 1.0},
            {'type': 'eq', 'fun': lambda x: np.sum(x[13:19]) - 1.0},
        ]

    def run_optimization(self):
        return minimize(
            self.objective,
            self.x0,
            method="trust-constr",
            bounds=self.get_bounds(),
            constraints=self.get_constraints(),
            options={
                "verbose": 2,
                "gtol": 1e-4,
                "xtol": 1e-4,
                "maxiter": 1000,
            }
        )
    
    def _set_eigenvalues(self, problem: Problem, x):
        t = self.t
        U = x[0]
        sublattice_labels = ["A", "B", "C", "D", "E", "F"]
        cell = problem.cell_parser
        g = cell.geometry
        n_subs = len(g.delta_vectors.value)
        subs = sublattice_labels[:n_subs]
        for i, label_i in enumerate(subs):
            parser = getattr(problem.cell_parser.eigenvalues, label_i).value
            # Diagonal Values
            if label_i == "A":
                n_px_up = x[1]
                n_px_down = x[2]
                n_py_up = x[3]
                n_py_down = x[4]
                n_pz_up = x[5]
                n_pz_down = x[6]
            elif label_i == "B":
                n_px_up = x[7]
                n_px_down = x[8]
                n_py_up = x[9]
                n_py_down = x[10]
                n_pz_up = x[11]
                n_pz_down = x[12]
            else:
                n_px_up = x[13]
                n_px_down = x[14]
                n_py_up = x[15]
                n_py_down = x[16]
                n_pz_up = x[17]
                n_pz_down = x[18]
            # Set parameters for all sublattices
            parser["onsite_energy"][label_i]["E_s"] = -5
            parser["interaction"][label_i]["U_p"] = U
            parser["interaction"][label_i]["n_px_up"] = n_px_up
            parser["interaction"][label_i]["n_px_down"] = n_px_down
            parser["interaction"][label_i]["n_py_up"] = n_py_up
            parser["interaction"][label_i]["n_py_down"] = n_py_down
            parser["interaction"][label_i]["n_pz_up"] = n_pz_up
            parser["interaction"][label_i]["n_pz_down"] = n_pz_down
            parser["chadi_soc"][label_i]["Delta_pp"] = self.Delta_SOC
            # Off-Diagonal Values
            for label_j in subs:
                # Hoppings
                try:
                    parser["nn_hopping"][label_j]["t_ss_sigma"] = 0.01 * np.abs(t)
                    parser["nn_hopping"][label_j]["t_pp_sigma"] = t
                    parser["nn_hopping"][label_j]["t_pp_pi"] = t/2
                except KeyError:
                    continue  # Skip if label_j not found

In [ ]:
optimizer = InteractionOptimizer()
res = optimizer.run_optimization()

print("Optimized occupation:", res.x)
print("Final RMSE:", res.fun)
print("Constraint sum:", np.sum(res.x))


ValueError: `constraint` of an unknown type is passed.